# Formative 1, Part 2 — Classical ML Classification Challenge
### Starter Notebook

**Your name(s):** _[fill in]_
**Kaggle username:** _[fill in]_
**Competition link:** _[paste the Kaggle competition URL here]_
**W&B project link:** _[paste once you've logged your first run]_

---

This notebook is a **template and a working baseline** — not a finished submission. It:
- Loads the competition data
- Explores it briefly
- Builds a simple, honest baseline (logistic regression), **fully logged to Weights & Biases**
- Generates a correctly-formatted submission file
- Gives you a reusable `log_experiment(...)` helper so every model you try afterward gets tracked the same way, with almost no extra code

**Your job:** keep this structure, replace/extend Section 5 onward with your own preprocessing decisions, additional models, and hyperparameter experiments — reusing `log_experiment(...)` for each one. Do not delete the baseline — your results table should include it as the reference point everything else is compared against.

**Remember:** you may be randomly selected to walk through your own W&B run history and explain it. Log everything, including runs that didn't work — a thin or fabricated-looking history is a problem in that session, not just for the rubric.


## 1. Introduction

_Replace this cell with 2-3 sentences: what is this competition, what are you predicting, and what metric are you optimizing? (See the competition Overview and Evaluation tabs.)_


## 2. Setup

Run this first. It installs/imports what you need and locates the data whether you're running in **Kaggle Notebooks**, **Google Colab**, or **locally**.


In [ ]:
import numpy as np
import pandas as pd
import os

pd.set_option("display.max_columns", 50)

# --- Locate the data automatically across common environments ---
CANDIDATE_DIRS = [
    "/kaggle/input",                 # Kaggle Notebooks (competition attached)
    "/content",                      # Google Colab (if you've uploaded/mounted files)
    ".",                             # local / same-folder
]

def find_data_dir():
    for base in CANDIDATE_DIRS:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if "train.csv" in files and "test.csv" in files:
                return root
    return None

DATA_DIR = find_data_dir()
if DATA_DIR is None:
    print("Could not auto-locate train.csv/test.csv.")
    print("If you're on Colab: upload the files or mount Drive, then set DATA_DIR manually below.")
    print("If you're on Kaggle: make sure you've clicked 'Add Data' and attached this competition's dataset.")
else:
    print(f"Found data in: {DATA_DIR}")


In [ ]:
# If auto-detection above failed, set the path manually and re-run:
# DATA_DIR = "/content"  # example for Colab after uploading files

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print("train shape:", train.shape)
print("test shape:", test.shape)
train.head()


## 3. Data Understanding

_Replace/extend this section with your own exploration. The baseline below does the bare minimum (shape, class balance, missingness) — you're expected to go further: distributions, correlations, a PCA or similar visualization, and a written interpretation of what you find. Remember: individual feature correlations with the target may look weak even where real signal exists — don't conclude "no signal" too quickly._


In [ ]:
target_col = "target"
feature_cols = [c for c in train.columns if c not in ("id", target_col)]
num_cols = [c for c in feature_cols if c.startswith("num_feat")]
cat_cols = [c for c in feature_cols if c.startswith("cat_")]

print(f"{len(num_cols)} numeric features, {len(cat_cols)} categorical features")
print("\nClass balance:")
print(train[target_col].value_counts(normalize=True).round(3))

print("\nMissingness (columns with any missing values):")
miss = train.isnull().mean()
print(miss[miss > 0].sort_values(ascending=False).round(3))


## 4. Experiment Tracking Setup (Weights & Biases)

Set this up **before** training anything, so your baseline run below is logged like every run after it.

Store your API key as a secret — never hardcode it in this notebook:
- **Google Colab:** left sidebar → key icon → add secret named `WANDB_API_KEY`
- **Kaggle Notebooks:** Add-ons menu → Secrets → add `WANDB_API_KEY`

This cell is written defensively: if W&B isn't set up yet, the rest of the notebook still runs — you just won't get tracking until you fix it. Don't leave it broken for long; ungraded/unlogged runs don't count.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

WANDB_ENABLED = False
WANDB_PROJECT = "formative1-part2-yourname"  # <-- change this

try:
    import wandb
    logged_in = False

    # --- Uncomment the block matching your environment, then set logged_in = True ---

    # Google Colab:
    # from google.colab import userdata
    # wandb.login(key=userdata.get("WANDB_API_KEY"))
    # logged_in = True

    # Kaggle Notebooks:
    # from kaggle_secrets import UserSecretsClient
    # wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    # logged_in = True

    WANDB_ENABLED = logged_in
    if WANDB_ENABLED:
        print("W&B ready. Runs will be logged to project:", WANDB_PROJECT)
    else:
        print("W&B installed but not logged in yet — uncomment the block above for your environment.")
        print("Until then, the notebook still runs, but NOTHING is being tracked.")
except Exception as e:
    print("W&B not available — runs will NOT be logged until you fix this.")
    print("Reason:", e)


### Reusable experiment logger

Every model you train from here on — baseline or otherwise — should go through this function. It fits your pipeline, evaluates it (holdout + cross-validation ROC-AUC), logs everything to W&B (config, metrics, confusion matrix), and keeps a local record you'll use to build your results table in Section 8.

You should not need to touch this function. Call it once per experiment with a descriptive `run_name` and a `config` dict describing what's different about that run.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

RANDOM_STATE = 42
results_log = []  # every experiment's summary lands here -> becomes your results table in Section 8

def log_experiment(run_name, pipeline, config, X_train, y_train, X_val, y_val, do_cv=True, X_full=None, y_full=None):
    '''
    Fits `pipeline`, evaluates it, logs to W&B (if enabled), and records
    a row for the results table. Returns the fitted pipeline.

    run_name : short descriptive string, e.g. "rf_depth8_lr0.05"
    config   : dict of whatever you want tracked, e.g. {"model": "RandomForest", "max_depth": 8}
    '''
    pipeline.fit(X_train, y_train)
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_probs)

    cv_mean, cv_std = None, None
    if do_cv and X_full is not None:
        cv_scores = cross_val_score(
            pipeline, X_full, y_full,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring="roc_auc",
        )
        cv_mean, cv_std = cv_scores.mean(), cv_scores.std()

    print(f"[{run_name}] validation ROC-AUC = {val_auc:.4f}" + (f", CV ROC-AUC = {cv_mean:.4f} (+/- {cv_std:.4f})" if cv_mean else ""))

    if WANDB_ENABLED:
        try:
            run = wandb.init(project=WANDB_PROJECT, name=run_name, config=config, reinit=True)
            log_dict = {"val_roc_auc": val_auc}
            if cv_mean is not None:
                log_dict.update({"cv_roc_auc_mean": cv_mean, "cv_roc_auc_std": cv_std})

            val_preds_hard = (val_probs >= 0.5).astype(int)
            cm = confusion_matrix(y_val, val_preds_hard)
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(cm, cmap="Blues")
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, str(cm[i, j]), ha="center", va="center")
            ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(run_name)
            wandb.log({**log_dict, "confusion_matrix": wandb.Image(fig)})
            plt.close(fig)
            run.finish()
        except Exception as e:
            print(f"  (W&B logging failed for this run: {e} — local results still recorded below)")

    results_log.append({
        "run_name": run_name,
        **config,
        "val_roc_auc": round(val_auc, 4),
        "cv_roc_auc_mean": round(cv_mean, 4) if cv_mean else None,
    })
    return pipeline


## 5. Preprocessing & Baseline Model

This section builds a **logistic regression baseline** end-to-end: impute missing values, scale numeric features, one-hot encode categoricals, then train and log it using `log_experiment(...)` from Section 4.

**This baseline is intentionally simple.** It is not meant to score well — it exists so you have:
1. A working, fully-tracked pipeline you can confirm end-to-end before building anything fancier.
2. A concrete number to beat, already sitting in `results_log` and on your W&B dashboard. If your "improved" model doesn't beat this, something is likely wrong, not just under-tuned.

Do not treat this as your final model. You are required to try at least one other model family (see the assignment instructions) and tune it properly — using the same `log_experiment(...)` pattern.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

X = train[feature_cols]
y = train[target_col]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

baseline_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

baseline_pipeline = log_experiment(
    run_name="baseline-logreg",
    pipeline=baseline_pipeline,
    config={"model": "LogisticRegression", "max_iter": 1000},
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    X_full=X, y_full=y,
)


## 6. Your Models & Experiments

**This is where your actual work goes.** Use the same `log_experiment(...)` pattern from Section 5 — fit a pipeline, call `log_experiment(run_name=..., pipeline=..., config={...}, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, X_full=X, y_full=y)` — for every model and every hyperparameter variation you try.

Add at least one classical model family beyond the logistic regression baseline (e.g., decision tree, random forest, SVM, XGBoost, LightGBM), justify your choice in Markdown, and log at least 3 hyperparameter variations per model as required by the assignment.


In [ ]:
# Your models go here. Reuse the pattern from Section 5, e.g.:
#
# from sklearn.ensemble import RandomForestClassifier
#
# my_pipeline = Pipeline([
#     ("prep", preprocess),
#     ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
# ])
#
# log_experiment(
#     run_name="descriptive-run-name",
#     pipeline=my_pipeline,
#     config={"model": "RandomForest", ...},
#     X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
#     X_full=X, y_full=y,
# )


## 7. Generating a Submission

This cell turns **any** fitted pipeline into a correctly-formatted submission file. It uses the baseline as the example — swap in your best model once you have one.

**Format check:** `sample_submission.csv` has two columns — `id` and `target` — where `target` is a **predicted probability**, not a 0/1 label. Using `predict_proba(...)[:, 1]` (not `predict(...)`) is what makes this correct.


In [ ]:
# Refit your chosen model on the FULL training set (not just the 80% split) before predicting on test
final_model = baseline_pipeline  # <-- replace with your actual best model once you have one
final_model.fit(X, y)
test_probs = final_model.predict_proba(test[feature_cols])[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "target": test_probs,
})

submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nSaved submission.csv — {submission.shape[0]} rows")


### How to submit this file to Kaggle

1. Go to the competition page → **Submit Predictions** (or the button on the Overview tab).
2. Upload `submission.csv` (download it first if you're on Colab: files panel → download; on Kaggle Notebooks it's already in your working directory).
3. Add a short description, e.g. `"baseline logistic regression"` or `"tuned random forest, run rf_n400_depth10"` — matching your W&B run name makes it easy to trace later.
4. Check your score on the **Leaderboard** tab once scoring finishes — this is your **public** score. Remember: your grade is based on the **private** leaderboard, revealed after the deadline. Don't over-optimize for the number you can see.
5. Repeat with your own models — you get up to 5 submissions/day, and can select up to 2 as your final submissions for grading.


## 8. Results Table

Auto-built from everything logged via `log_experiment(...)` above — this should already contain your baseline and every experiment you ran. Add a `public_lb_score` and `wandb_run_url` column by hand once you've submitted to Kaggle and can see your W&B run links.


In [ ]:
results_df = pd.DataFrame(results_log)
results_df["public_lb_score"] = None   # fill in after submitting to Kaggle
results_df["wandb_run_url"] = None     # paste your W&B run links here
results_df


## 9. Discussion

_Replace with your written analysis (academic prose, not just bullet fragments): which models and choices mattered most, why you think that is, what you'd try next with more time, and what your experiments taught you about this dataset. Reference specific rows from your results table and specific W&B runs._


## 10. References

_List the competition/dataset link, and any papers, articles, or documentation you cited, in a consistent format (APA or IEEE)._
